In [9]:
#imports 
import rasterio
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier

## Importing data

In [10]:
back = rasterio.open("/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/backscatter.tif")
bath = rasterio.open("/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/bathymetry.tif")
train_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/test.csv')

In [11]:
print('Train set: ', train_df.shape)
display(train_df.head())

print('Test set: ', test_df.shape)
display(test_df.head())

back_data = back.read(1)
print('Backscatter data: ',back_data.shape)
print(back_data[:5, :5])

bath_data = bath.read(1)
print('Bathymestry data: ', bath_data.shape)
print(bath_data[:5, :5])

Train set:  (6256, 3)


,class,x,y
0,NVB,453594.477237,5.679192e+06
1,FMAT,453561.906453,5.679109e+06
2,ALG,453744.452238,5.679033e+06
3,ALG,453863.445302,5.679038e+06
4,ALG,453964.611906,5.679017e+06


Test set:  (98, 3)


,ID,x,y
0,1,453702.166779,5.679044e+06
1,2,454126.252800,5.678999e+06
2,3,453957.881092,5.678942e+06
3,4,453798.917484,5.678955e+06
4,5,453520.953671,5.679124e+06


Backscatter data:  (4040, 4743)
[[-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]]
Bathymestry data:  (4040, 4743)
[[-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]]


## Extracting Features

In [12]:
def get_features(x, y):
    row, col = bath.index(x, y)
    return bath_data[row, col], back_data[row, col]

train_df[['depth', 'scatter']] = train_df.apply(
    lambda r: pd.Series(get_features(r['x'], r['y'])),
    axis=1
)

test_df[['depth', 'scatter']] = test_df.apply(
    lambda r: pd.Series(get_features(r['x'], r['y'])),
    axis=1
)

### Neighborhood features:
Instead of this:

     X   ← your point

You look at:

      □ □ □
      □ X □   ← n x n grid
      □ □ □
This gives context

#### 3x3 trial:

In [13]:
def get_neighborhood_features(x, y):
    row, col = bath.index(x, y)
    
    features = []
    
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            r = row + dr
            c = col + dc
            
            try:
                depth = bath_data[r, c]
                scatter = back_data[r, c]
            except:
                depth = 0
                scatter = 0
            
            features.append(depth)
            features.append(scatter)
    
    return features

In [14]:
features = train_df.apply(
    lambda r: get_neighborhood_features(r['x'], r['y']),
    axis=1
)

train_feature_cols = [f'f{i}' for i in range(len(features.iloc[0]))]

train_df[train_feature_cols] = pd.DataFrame(features.tolist(), index=train_df.index)

In [15]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6256 entries, 0 to 6255
Data columns (total 23 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   class    6256 non-null   object 
 1   x        6256 non-null   float64
 2   y        6256 non-null   float64
 3   depth    6256 non-null   float32
 4   scatter  6256 non-null   float32
 5   f0       6256 non-null   float32
 6   f1       6256 non-null   float32
 7   f2       6256 non-null   float32
 8   f3       6256 non-null   float32
 9   f4       6256 non-null   float32
 10  f5       6256 non-null   float32
 11  f6       6256 non-null   float32
 12  f7       6256 non-null   float32
 13  f8       6256 non-null   float32
 14  f9       6256 non-null   float32
 15  f10      6256 non-null   float32
 16  f11      6256 non-null   float32
 17  f12      6256 non-null   float32
 18  f13      6256 non-null   float32
 19  f14      6256 non-null   float32
 20  f15      6256 non-null   float32
 21  f16      6256 

In [16]:
target = 'class'

X = train_df[train_feature_cols]
y = train_df[target].map({'SGAM': 0, 'NVB': 1, 'SGZ': 2, 'ALG': 3, 'FMAT': 4})

## Model:

In [17]:
#splitting the data into train/val splits
X_train , X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=24)

In [18]:
model = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [19]:
val_preds = model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
val_f1 = f1_score(y_val, val_preds, average='weighted')

print('Validation accuracy: ', val_acc)
print('Validation f1: ', val_f1)

Validation accuracy:  0.7675718849840255
Validation f1:  0.7652371112336698


### 5x5 trial:
Increasing the search area

In [29]:
def get_neighborhood_features(x, y):
    row, col = bath.index(x, y)
    
    features = []
    
    for dr in [-3, 0, 3]:
        for dc in [-3, 0, 3]:
            r = row + dr
            c = col + dc
            
            try:
                depth = bath_data[r, c]
                scatter = back_data[r, c]
            except:
                depth = 0
                scatter = 0
            
            features.append(depth)
            features.append(scatter)
    
    return features

In [30]:
features = train_df.apply(
    lambda r: get_neighborhood_features(r['x'], r['y']),
    axis=1
)

train_feature_cols = [f'f{i}' for i in range(len(features.iloc[0]))]

train_df[train_feature_cols] = pd.DataFrame(features.tolist(), index=train_df.index)

In [31]:
X = train_df[train_feature_cols]
y = train_df[target].map({'SGAM': 0, 'NVB': 1, 'SGZ': 2, 'ALG': 3, 'FMAT': 4})

In [32]:
#splitting the data into train/val splits
X_train , X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=24)

In [33]:
model = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [34]:
val_preds = model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
val_f1 = f1_score(y_val, val_preds, average='weighted')

print('Validation accuracy: ', val_acc)
print('Validation f1: ', val_f1)

Validation accuracy:  0.8322683706070287
Validation f1:  0.8316638029256713


7x7 surrounding grid gives more information than 3x3 as can be seen through validation scores, also if you think about it it does make sense. The more surrounding area you have information on teh better you will be able to predict.

## Predictions:

In [36]:
features = test_df.apply(
    lambda r: get_neighborhood_features(r['x'], r['y']),
    axis=1
)

test_feature_cols = [f'f{i}' for i in range(len(features.iloc[0]))]

test_df[test_feature_cols] = pd.DataFrame(features.tolist(), index=test_df.index)

In [37]:
X_test = test_df[test_feature_cols]

In [38]:
y_preds = model.predict(X_test)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    target: y_preds
})

submission[target] = submission[target].map({0: 'SGAM', 1: 'NVB', 2: 'SGZ', 3: 'ALG', 4: 'FMAT'})

submission.to_csv('submission.csv', index=False)